# 🔤 NLP Preprocessing Engine – Data Science Internship Feb 2026
### Task 1: Build a Robust NLP Preprocessing Engine (Advanced)
---

## 📚 Task 1: Conceptual Understanding (Written Answers)

### Q1. What is the difference between 'Love' and 'love' in NLP?

In NLP, **'Love'** and **'love'** are treated as two different tokens by default because most NLP systems are case-sensitive. A word at the start of a sentence is capitalized ('Love'), but it carries the same semantic meaning as its lowercase form ('love'). To ensure the model treats them as the same word, we apply **lowercasing** as a preprocessing step. Without it, the model may learn different representations for the same word, causing inconsistency and increasing vocabulary size unnecessarily.

---

### Q2. What happens if stopwords are not removed?

Stopwords are common words like 'is', 'the', 'a', 'and', etc. If not removed:
- They dominate frequency counts and bias models toward irrelevant tokens.
- They increase model complexity without adding meaningful information.
- They can reduce accuracy in tasks like text classification or sentiment analysis.
- They slow down training due to larger vocabulary size.

---

### Q3. Two real-world scenarios where removing stopwords can be HARMFUL:

1. **Sentiment Analysis with Negations**: Removing stopwords like 'not', 'no', 'never' completely flips the sentiment. For example, *'I am not happy'* becomes *'happy'* after removing stopwords — the exact opposite meaning.

2. **Question Answering / Chatbots**: In a query like *'Who is the CEO of Apple?'*, removing 'who', 'is', 'the' leaves *'CEO Apple'* — which loses the question structure, making intent classification much harder.

---

### Q4. Difference between Stemming and Lemmatization:

| Feature | Stemming | Lemmatization |
|---|---|---|
| Method | Chops off word endings using rules | Uses dictionary/vocabulary to find root form |
| Output | May not be a real word | Always a valid dictionary word |
| Speed | Faster | Slower (needs POS tagging) |
| Example | 'running' → 'run' or 'runn' | 'running' → 'run' |
| Example | 'studies' → 'studi' | 'studies' → 'study' |
| Accuracy | Lower | Higher |

---
## ⚙️ Setup & Imports

In [ ]:
# Install required libraries (run this if using Google Colab)
# !pip install nltk

import re
import string
from collections import Counter
import nltk

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.corpus import stopwords

print("✅ All libraries loaded successfully!")

---
## 🛠️ Task 2: Build Advanced Preprocessing Function

In [ ]:
# Words to preserve even if they are short (length <= 2)
MEANINGFUL_SHORT_WORDS = {'no', 'not', 'ok', 'go', 'do', 'an', 'my', 'me', 'us', 'we'}

# Standard English stopwords
STOPWORDS = set(stopwords.words('english'))

# Keep negations — do not treat them as stopwords
PRESERVE_FROM_STOPWORDS = {'no', 'not', 'nor', 'neither', 'never', 'none'}
STOPWORDS -= PRESERVE_FROM_STOPWORDS


def preprocess_text(text):
    """
    Advanced NLP Preprocessing Function.
    
    Steps:
    1. Handle None / non-string edge cases
    2. Convert to lowercase
    3. Remove URLs and email-like patterns
    4. Remove emojis and non-ASCII characters
    5. Remove punctuation and special characters
    6. Remove numbers
    7. Handle repeated characters (e.g., 'soooo' → 'so')
    8. Tokenize
    9. Remove very short tokens (length <= 2), keeping meaningful ones
    10. Remove extra spaces
    
    Args:
        text (str): Raw input text
    
    Returns:
        tuple: (list of tokens, cleaned sentence string)
    """
    
    # --- Error Handling ---
    if text is None:
        return [], ""
    
    if not isinstance(text, str):
        text = str(text)
    
    # Handle edge cases: empty string, only emojis, only numbers
    stripped = text.strip()
    if not stripped:
        return [], ""  # Empty string
    
    # --- Step 1: Convert to lowercase ---
    text = text.lower()
    
    # --- Step 2: Remove URLs (http, https, www) ---
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # --- Step 3: Remove email-like patterns ---
    text = re.sub(r'\S+@\S+', '', text)
    
    # --- Step 4: Remove emojis and non-ASCII characters ---
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # --- Step 5: Remove punctuation and special characters ---
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # --- Step 6: Handle repeated characters (sooo → so, baaad → bad) ---
    # Replace any character repeated more than 2 times with just 2 occurrences
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Then reduce double chars to single for common cases (e.g., 'haa' → 'ha' may be needed)
    # We use a gentle approach: reduce to single only for obvious repeats
    text = re.sub(r'(\w)\1+', r'\1', text)
    
    # --- Step 7: Tokenize by splitting on whitespace ---
    tokens = text.split()
    
    # --- Step 8: Remove very short tokens (length <= 2), except meaningful words ---
    tokens = [
        token for token in tokens
        if len(token) > 2 or token in MEANINGFUL_SHORT_WORDS
    ]
    
    # --- Step 9: Remove extra spaces (handled by split + join) ---
    clean_sentence = ' '.join(tokens)
    
    return tokens, clean_sentence


print("✅ preprocess_text() function defined!")

# --- Quick sanity checks ---
print("\n--- Quick Tests ---")
print(preprocess_text("I have 2 dogs"))          # Remove numbers
print(preprocess_text("This  is   good"))         # Remove extra spaces
print(preprocess_text("soooo goooood!!!"))         # Repeated chars
print(preprocess_text("WOW!!! This is GREAT!!!")) # Lowercase
print(preprocess_text("Visit http://example.com now"))  # Remove URL
print(preprocess_text("I am not happy with this"))  # Keep 'not'

---
## 🧪 Task 3: Stress Testing – 10 Diverse Sentences

In [ ]:
# Sample input sentences covering emojis, numbers, slang, repeated chars, URLs, mixed case
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print("=" * 70)
print("STRESS TESTING RESULTS")
print("=" * 70)

results = []  # Store results for later analysis

for i, sentence in enumerate(sample_inputs, 1):
    tokens, clean_sentence = preprocess_text(sentence)
    results.append({
        'original': sentence,
        'tokens': tokens,
        'clean_sentence': clean_sentence
    })
    
    print(f"\n[Sentence {i}]")
    print(f"  📌 Original Text    : {sentence}")
    print(f"  🔤 Cleaned Tokens   : {tokens}")
    print(f"  ✅ Cleaned Sentence : {clean_sentence}")
    print("-" * 70)

---
## 📊 Task 4: Token Analytics

In [ ]:
print("=" * 70)
print("TOKEN ANALYTICS")
print("=" * 70)

most_noisy_sentence = None
most_noisy_score = -1
most_meaningful_sentence = None
most_meaningful_count = -1

for i, result in enumerate(results, 1):
    tokens = result['tokens']
    original = result['original']
    
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_token_length = round(sum(len(t) for t in tokens) / total_tokens, 2) if tokens else 0
    
    # Noise = characters removed compared to original
    original_word_count = len(original.split())
    noise_score = original_word_count - total_tokens  # Words lost = noise
    
    if noise_score > most_noisy_score:
        most_noisy_score = noise_score
        most_noisy_sentence = (i, original)
    
    if total_tokens > most_meaningful_count:
        most_meaningful_count = total_tokens
        most_meaningful_sentence = (i, original)
    
    print(f"\n[Sentence {i}] {original[:50]}..." if len(original) > 50 else f"\n[Sentence {i}] {original}")
    print(f"  Total Tokens      : {total_tokens}")
    print(f"  Unique Tokens     : {unique_tokens}")
    print(f"  Avg Token Length  : {avg_token_length}")

print("\n" + "=" * 70)
print("📢 ANALYSIS QUESTIONS:")
print(f"  🔴 Most Noisy Sentence      → Sentence {most_noisy_sentence[0]}: '{most_noisy_sentence[1]}'")
print(f"  🟢 Most Meaningful Sentence → Sentence {most_meaningful_sentence[0]}: '{most_meaningful_sentence[1]}'")
print("=" * 70)

---
## 📈 Task 5: Frequency Analysis

In [ ]:
from collections import Counter

# Combine all tokens from all sentences
all_tokens = []
for result in results:
    all_tokens.extend(result['tokens'])

print(f"Total tokens across all sentences: {len(all_tokens)}")
print(f"Unique tokens across all sentences: {len(set(all_tokens))}")

token_counts = Counter(all_tokens)

print("\n" + "=" * 50)
print("🔝 Top 10 Most Frequent Words:")
print("=" * 50)
for word, count in token_counts.most_common(10):
    print(f"  '{word}' → {count} time(s)")

print("\n" + "=" * 50)
print("🔻 Top 5 Least Frequent Words:")
print("=" * 50)
least_common = token_counts.most_common()[-5:]
for word, count in least_common:
    print(f"  '{word}' → {count} time(s)")

---
## 🔄 Task 6: Build Full Pipeline

In [ ]:
def full_pipeline(text_list):
    """
    Full NLP Preprocessing Pipeline.
    
    Processes a list of raw text strings through the preprocessing engine
    and returns a structured dictionary with tokens and cleaned sentences.
    
    Args:
        text_list (list): List of raw text strings
    
    Returns:
        dict: {
            'tokens': list of token lists (one per sentence),
            'clean_sentences': list of cleaned sentence strings
        }
    """
    if not isinstance(text_list, list):
        raise TypeError("Input must be a list of strings.")
    
    all_tokens = []
    clean_sentences = []
    
    for text in text_list:
        tokens, clean_sentence = preprocess_text(text)
        all_tokens.append(tokens)
        clean_sentences.append(clean_sentence)
    
    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }


# Run the full pipeline on sample inputs
pipeline_output = full_pipeline(sample_inputs)

print("=" * 60)
print("FULL PIPELINE OUTPUT")
print("=" * 60)
print("\n📦 Tokens (per sentence):")
for i, toks in enumerate(pipeline_output['tokens'], 1):
    print(f"  [{i}] {toks}")

print("\n✅ Cleaned Sentences:")
for i, sent in enumerate(pipeline_output['clean_sentences'], 1):
    print(f"  [{i}] {sent}")

---
## 🛡️ Task 7: Error Handling

In [ ]:
print("=" * 60)
print("ERROR HANDLING TEST CASES")
print("=" * 60)

edge_cases = [
    ("",           "Empty String"),
    ("😍😂🔥💯",   "Only Emojis"),
    ("1234567890", "Only Numbers"),
    (None,         "None Input"),
    ("   ",        "Only Spaces"),
    ("!!!",        "Only Punctuation"),
]

for text, case_name in edge_cases:
    try:
        tokens, clean = preprocess_text(text)
        print(f"\n🧪 Case: {case_name}")
        print(f"   Input   : {repr(text)}")
        print(f"   Tokens  : {tokens}")
        print(f"   Output  : '{clean}'")
        if not tokens:
            print("   ⚠️  Result: Empty output (as expected for noisy/invalid input)")
    except Exception as e:
        print(f"\n❌ Case: {case_name} → Error: {e}")

print("\n" + "=" * 60)
print("✅ All edge cases handled gracefully!")

---
## 🎉 Summary

In this notebook, we successfully built a **Robust NLP Preprocessing Engine** that:

| Feature | Status |
|---|---|
| Lowercase conversion | ✅ |
| URL & email removal | ✅ |
| Emoji removal | ✅ |
| Number removal | ✅ |
| Repeated character handling | ✅ |
| Short token removal (keeps 'no', 'not') | ✅ |
| Extra space removal | ✅ |
| Token analytics | ✅ |
| Frequency analysis | ✅ |
| Full pipeline | ✅ |
| Error handling (empty, emoji-only, number-only) | ✅ |